# Audio Classification with RNN / LSTM (PyTorch)

This Colab notebook walks you through a **complete sequence-modeling pipeline for audio classification** using PyTorch:
- Create a **custom `Dataset`** class for audio + features (log-mel spectrograms)
- Define **RNN/LSTM** model classes (configurable)
- Implement **training & validation** loops with **accuracy** and **cross-entropy loss**
- **Tune hyperparameters** (simple grid) if desired
- **Plot** training/validation curves
- Build a **test** evaluator with **Confusion Matrix** and **Classification Report**


**The label for this lab is the speaker. In essence the goal is to predict who is speaking.**

## Setup


In [ ]:
!pip -q install torchaudio librosa soundfile scikit-learn


In [ ]:
import os, math, random, time, json, itertools, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import librosa
import soundfile as sf
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torch.optim as optim
from tqdm import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

## Define the arguments needed to instantiate the MyCustomDataset class


We begin by loading the `label.csv` metadata file. This file contains the filename, label. The files are stored in the folder recording.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


We will be using the log-mel spectrogram transformation of torchaudio

SAMPLE_RATE = 8000
NUM_SAMPLES = 8000

In [ ]:
SAMPLE_RATE =
NUM_SAMPLES =
TARGET_MS = NUM_SAMPLES * 1000 // SAMPLE_RATE

ANNOTATIONS_FILE =
AUDIO_DIR =

In [ ]:
import pandas as pd

#read annotation

# Preview the first few rows


BE CAREFUL THE LABEL IS SPEAKER....

In [ ]:
annotation_df = pd.read_csv(ANNOTATIONS_FILE)  # has columns: filename, speaker
speakers =           # unique speaker names
num_classes = len()

# Transform speaker name into a unique number id


## Define Transforms (torchaudio)


We use torchaudio to define our transformation pipeline:
- Convert waveform into Mel Spectrogram
- Convert amplitude to decibels



In [ ]:
mel_spect =

log_mel_spect =

transform =

## Custom `Dataset`

Modify the Dataset class of last lab to suit the new data to be used for this lab

In [ ]:
class AudioUtil:
    def open(self, audio_file):
        sig, sr = torchaudio.load(audio_file)
        return (sig, sr)

    def rechannel(self, aud, new_channel=1):


    def resample(self, aud, newsr):


    def pad_trunc(self, aud, max_ms):



class MyCustomDataset(Dataset):
    def __init__(self, annotation_df, audio_dir, transform, sample_rate=8000,
                 max_ms=8000, num_channels=1, device="cpu"):


    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        file_name = self.annotations.iloc[index, 0]
        audio_path =
        label =

        signal, sr =
        #processing


        # Transform to spectrogram


        return signal,spec, label

### Instantiate an object if the dataset class

In [ ]:
dataset = MyCustomDataset(
    audio_dir=
    sample_rate=
    max_ms=
    transform=
    device=
    annotation_df=
)


### Use the len function to show the number of samples in the datasset


### Now that you have a custom-built dataset class, index the first sample in the dataset.
##### Print the shape of the signal and explain the result.
##### This should be followed by a plot of the log-mel spectrogram using librosa.display.specshow

In [ ]:
import librosa
import librosa.display

signal,logmel, label = dataset[0]



# ---- Plot log-mel with librosa.display.specshow ----


### Repeat the above for the last sample in the dataset

### Split the dataset into training (80%), validation(10%), and test (10%) dataset
##### Use the random_split from torch.utils.data

In [ ]:
full_dataset =

from torch.utils.data import random_split

# Total number of samples
dataset_size = len(dataset)
print(f"Total samples in dataset: {dataset_size}")

# Define split sizes
train_size =
val_size   =
test_size  =

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])


print(f"Train: {len(train_ds)} samples")
print(f"Validation: {len(val_ds)} samples")
print(f"Test: {len(test_ds)} samples")


Total samples in dataset: 3000
Train: 2400 samples
Validation: 300 samples
Test: 300 samples


### Create a train, validation and test dataloaders using dataloaders from torch.utils.data
##### Use a batch size of 64
##### Remeber to turn on shuffle for training and turn it off for validation and testing

In [ ]:
BATCH_SIZE = 64

# Create DataLoaders


## Model Classes — RNN and LSTM


### Create a RNN model class that contains the init and the forward methods
##### The model structure will be unique for every student
##### The model should consist of lstm and linear layers
##### Recall that there are 6 classes in the dataset, hence the final layer will have 6 neurons

FOR RNN:


- input_size – The number of expected features in the input x

- hidden_size – The number of features in the hidden state h

- num_layers – Number of recurrent layers. E.g., setting num_layers=2 would mean stacking two RNNs together to form a stacked RNN, with the second RNN taking in outputs of the first RNN and computing the final results. Default: 1

- nonlinearity – The non-linearity to use. Can be either 'tanh' or 'relu'. Default: 'tanh'

- bias – If False, then the layer does not use bias weights b_ih and b_hh. Default: True

- batch_first – If True, then the input and output tensors are provided as (batch, seq, feature) instead of (seq, batch, feature). Note that this does not apply to hidden or cell states. See the Inputs/Outputs sections below for details. Default: False

- dropout – If non-zero, introduces a Dropout layer on the outputs of each RNN layer except the last layer, with dropout probability equal to dropout. Default: 0

- bidirectional – If True, becomes a bidirectional RNN. Default: False

FOR LSTM:

- input_size – The number of expected features in the input x

- hidden_size – The number of features in the hidden state h

- num_layers – Number of recurrent layers. E.g., setting num_layers=2 would mean stacking two LSTMs together to form a stacked LSTM, with the second LSTM taking in outputs of the first LSTM and computing the final results. Default: 1

- bias – If False, then the layer does not use bias weights b_ih and b_hh. Default: True

- batch_first – If True, then the input and output tensors are provided as (batch, seq, feature) instead of (seq, batch, feature). Note that this does not apply to hidden or cell states. See the Inputs/Outputs sections below for details. Default: False

- dropout – If non-zero, introduces a Dropout layer on the outputs of each LSTM layer except the last layer, with dropout probability equal to dropout. Default: 0

- bidirectional – If True, becomes a bidirectional LSTM. Default: False

- proj_size – If > 0, will use LSTM with projections of corresponding size. Default: 0

In [ ]:

class RNNClassifier(nn.Module):
    def __init__(self, input_size=64, hidden_size=128, num_layers=1,
                 num_classes=6, model_type='rnn', dropout=0.0, bidirectional=False):
        self.model_type = model_type
        super().__init__()
        # RNN
        if self.model_type == 'rnn':

        #elif model is lstm
        elif self.model_type == 'lstm':

        else:
            raise ValueError("model_type must be either 'rnn' or 'lstm'")

        # Compute output dimension (doubles if bidirectional=True)
        out_dim =

        # Fully connected layer: maps hidden output -> class scores
        self.fc = nn.Linear(out_dim, num_classes)

    def forward(self, x):
        # Transpose to (batch, time, n_mels) for RNN/LSTM
        x = x.transpose(1, 2)

        # Forward pass through RNN or LSTM
        if self.model_type == 'rnn':
            rnn_out, h_n = self.recurrent(x)
        else:
            rnn_out, (h_n, c_n) = self.recurrent(x)
        out = rnn_out.mean(dim=1)         # temporal mean pooling
        out = self.fc(out)
        return out


### Use the summary fuction in the torchinfo library to print the number of parameters and the estimated size of the model RNN and LSTM

### Make a comparison of the number of parameters and explain why they  are different

In [ ]:
!pip install torchinfo



In [ ]:
from torchinfo import summary
rnn_model =
lstm_model =

# Move models to the same device
rnn_model.to(device)
lstm_model.to(device)
summary(rnn_model,  input_size=(1, 64, 50))


Layer (type:depth-idx)                   Output Shape              Param #
RNNClassifier                            [1, 6]                    --
├─RNN: 1-1                               [1, 50, 128]              24,832
├─Linear: 1-2                            [1, 6]                    774
Total params: 25,606
Trainable params: 25,606
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 1.24
Input size (MB): 0.01
Forward/backward pass size (MB): 0.05
Params size (MB): 0.10
Estimated Total Size (MB): 0.17

In [ ]:
summary(lstm_model, input_size=(1, 64, 50))

Layer (type:depth-idx)                   Output Shape              Param #
RNNClassifier                            [1, 6]                    --
├─LSTM: 1-1                              [1, 50, 128]              99,328
├─Linear: 1-2                            [1, 6]                    774
Total params: 100,102
Trainable params: 100,102
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 4.97
Input size (MB): 0.01
Forward/backward pass size (MB): 0.05
Params size (MB): 0.40
Estimated Total Size (MB): 0.46

LSTMs have 4 times more parameters than RNNs
because they include four gates (input, forget, cell, output)
each with its own weight matrices and biases —
giving them better control over memory, but making them heavier.

### Create a Training Function for one epoch only
##### Using a for loop, the train function will take as arguments: the model, train loader, loss function, optimizer, etc
##### Use the cross-entropy loss function and calculate accuracy

In [ ]:

def train_one_epoch(model, loader, loss_fn, optimizer, device):

    acc = correct / total
    avg_loss = total_loss / len(loader)

    print(f"Train Accuracy: {acc:.4f} | Loss: {avg_loss:.4f}")
    return avg_loss, acc

### Run it for 1 epoch

### Create a Validation Function for one epoch only
##### Using a for loop, the validation function will take as arguments: the model, validation loader, loss function
##### It will be wise to include best model parameters are arguments at this point
##### The goal is to monitor the performance of your proposed model to avoid overfitting/underfitting, know the best epoch to save the model weights, etc.

### VALIDATE FOR 1 EPOCH

### Create a Function that will call the training and validation functions just created
##### This function will interate over all the epochs that you define
##### It will also activate your early stopping, so that the model does not continue training without improvements in the validation

# 4) Now commence training.


### Normally, you should manually revisit the models function to tune the hyperparameters with a goal of improving the results
##### Some of the hyperparameters to be tuned include
###### Learning rate
###### Number of RNN/LSTM layers
###### Number of neurons in the fully connected layer

**HERE YOU WILL FIX THE HYPERPARAMETERS**

**IF YOU HAVE TIME: Try only to pick one hyperparameter and try 2 values different values**

### save the model weights for RNN and LSTM

In [ ]:
INPUT_SIZE   = 64          # n_mels
HIDDEN_SIZE  = 128
NUM_LAYERS   = 4
BATCH_SIZE   = 32
EPOCHS       = 10
PATIENCE     = 3
LEARNING_RATE = 1e-3
NUM_CLASSES=6
DROPOUT=0.2

In [ ]:

# RNN model
rnn_model = RNNClassifier(

).to(device)

# LSTM model
lstm_model = RNNClassifier(

).to(device)

loss_fn =
opt_rnn  =
opt_lstm =


In [ ]:
best_rnn_model, rnn_train_losses, rnn_val_losses, rnn_train_accs, rnn_val_accs = full_train(
    rnn_model, train_loader, val_loader,
    loss_fn, opt_rnn, device,
    num_epochs=EPOCHS, patience=PATIENCE
)

torch.save(best_rnn_model, '/content/drive/MyDrive/FSDD/best_rnn_weights.pth')

print("Saved: best_rnn_weights.pth")

In [ ]:
# --- LSTM ---
best_lstm_model, lstm_train_losses, lstm_val_losses, lstm_train_accs, lstm_val_accs = full_train(
    lstm_model, train_loader, val_loader,
    loss_fn, opt_lstm, device,
    num_epochs=EPOCHS, patience=PATIENCE
)

torch.save(best_lstm_model, '/content/drive/MyDrive/FSDD/best_lstm_weights.pth')
print(" Saved: best_lstm_weights.pth")

### Make a plot of the training and validation losses as well as the accuracies over all the epochs considered

In [ ]:

# Load the best model weights
rnn_model.load_state_dict(torch.load('/content/drive/MyDrive/FSDD/best_rnn_weights.pth'))
rnn_model.eval()  # set to evaluation model
plot_training_curves(rnn_train_losses, rnn_val_losses, rnn_train_accs, rnn_val_accs)

In [ ]:
# Load the best model weights
lstm_model.load_state_dict(torch.load('/content/drive/MyDrive/FSDD/best_lstm_weights.pth'))
lstm_model.eval()  # set to evaluation model
plot_training_curves(lstm_train_losses, lstm_val_losses, lstm_train_accs, lstm_val_accs)

#TEST

### Create a Test function
##### This will use the best models weights over the test data
##### The performance on the test data should be similar to the performance recorded at the best epoch during training and validation

In [ ]:
def test_one_epoch(model, val_loader, loss_fn, device):


In [ ]:
rnn_model.load_state_dict(torch.load('/content/drive/MyDrive/FSDD/best_rnn_weights.pth'))
rnn_model.eval()
rnn_test_loss, rnn_test_acc,rnn_pred,rnn_true = test_one_epoch(rnn_model, test_loader, loss_fn, device)

In [ ]:
lstm_model.load_state_dict(torch.load('/content/drive/MyDrive/FSDD/best_lstm_weights.pth'))
lstm_model.eval()
test_loss, test_acc,lstm_pred,lstm_true = test_one_epoch(lstm_model, test_loader, loss_fn, device)

### Evaluate the performance using Confusion Matrix and Classification Report
##### The sklearn library provides functions that can be used to visualize these results
